# **Practical 6**
Welcome to the last practical for Graph Representation Learning (MT22).

We will be comparing GNNs with the 1-WL hash algorithm, following the work done in practical 5.

The notebook is divided into sections, each of which comes with complete or partially completed code. Before each snippet of code there will be a description of what we are about to implement. The sections of code you need to complete are marked as Tasks.

Please ensure that you operate within the framework given in the notebook and bring any questions you may have to the practical demonstrators. We suggest that you DO NOT edit code that is a part of the framework, since this will make it more difficult for demonstrators to assist if your code is broken.

In [1]:
!pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/, https://download.pytorch.org/whl/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.8/1.8 GB 57.1 MB/s eta 0:00:01tcmalloc: large alloc 1837744128 bytes == 0x27de000 @  0x7fb5b7c501e7 0x4d30a0 0x4d312c 0x5d6f4c 0x51edd1 0x51ef5b 0x4f750a 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x5d8868 0x4997a2 0x55cd91 0x5d8941 0x49abe4 0x55cd91 0x5d8941 0x4997a2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.8/1.8 GB 58.1 MB/s eta 0:00:01tcmalloc: large alloc 2297184256 bytes == 0x7007a000 @  0x7fb5b7c51615 0x5d6f4c 0x51edd1 0x51ef5b 0x4f750a 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941 0x4997a2 0x5d8868 0x4997a2 0x55cd91 0x5d8941 0x49abe4 0x55cd91 0x5d8941 0x4997a2 0x55cd91 0x5d8941
     ━━━━━━━━━━━━━━━━━━━

In [ ]:
# Check PyTorch version installed on this system
!python -c "import torch; print(torch.__version__)"

1.12.1+cu113


In [ ]:
%%capture
# Download the corresponding PyTorch Geometric module
"""
Assign to TORCH with what you get from the cell above. E.g., export TORCH=1.12.1+cu113
"""
%env TORCH=1.12.1+cu113
!pip install torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install torch-geometric

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader
from torch_geometric.nn import Sequential, GCNConv, global_mean_pool
import networkx as nx
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.default_rng()

# **Part 0: Importing results from Practical 5**
Generate graph pairs as required by Task 1.1 of P5. You can simply copy your code and paste it below.

In [ ]:
# The range of different graph sizes
size_range = range(6, 16)

# The list of all graph pairs
graph_pairs = []

for n in size_range:
    c = nx.cycle_graph(n)
    for i in range(3, n - 2):
        d = nx.union(nx.cycle_graph(n - i), nx.cycle_graph(i), rename=("A", "B"))
        graph_pairs.append((c, d))

# **Part 1: Building a GNN**

## **Task 1.1: Converting to PyTorch Geometric objects**

Let's turn each graph into `torch_geometric.data.Data` objects, with some input features `x` and output features `y`.
Each previously generated pair should be converted to two Data objects, one for the cycle graph and one for the disjoint union one. The features `x` corresponds to the labels of each node; we will use the same label for each node (e.g., an array of 50 elements each set to 1). The output `y` should be 1 for cycle graphs and 0 for disjoint graphs.

(Look at the function `from_networkx`)

In [ ]:
# The list of Data objects
dataset = []

# Loop over graph pairs
for pair in graph_pairs:
    # Treat each element of the pair
    for i, graph in enumerate(pair):
        graph_geom = from_networkx(graph)
        n = graph_geom.num_nodes
        graph_geom.x = torch.ones(n, 50)
        if i == 0:
            graph_geom.y = torch.ones(1).type(torch.LongTensor)
        else:
            graph_geom.y = torch.zeros(1).type(torch.LongTensor)
        # Add the Data object to the dataset
        dataset.append(graph_geom)

print(dataset[0])
print(dataset[1])
print(len(dataset))

Data(edge_index=[2, 12], num_nodes=6, x=[6, 50], y=[1])
Data(edge_index=[2, 12], num_nodes=6, x=[6, 50], y=[1])
110


## **Part 1.2: Defining the model**
Let's define our graph neural network. First, we define a module for doing mean pooling.

In [ ]:
class GlobalMeanPool(nn.Module):
    """Global mean pool layer."""

    def forward(self, x, batch=None, size=None):

        # If we don't get the batch vector, set it to zeros.
        if batch is None:
            batch = x.new_zeros(x.size(0), dtype=torch.int64)

        return global_mean_pool(x, batch, size)

This is the base GNN model. The function _layer_sequence specifies the list of modules which define the model.

In [ ]:
class GraphSequenceModel(nn.Module):
    """A GNN consisting of a stack of layers."""

    def __init__(self, *args, **kwargs):
        super().__init__()
        sequence = self._layer_sequence(*args, **kwargs)
        self.stack = Sequential("x, edge_index, batch", sequence)

    def forward(self, batch):
        return self.stack.forward(batch.x, batch.edge_index, batch.batch)

    @staticmethod
    def _weight_reset(module):
        if isinstance(module, GCNConv) or isinstance(module, nn.Linear):
            module.reset_parameters()

    def reset_parameters(self):
        return self.stack.apply(type(self)._weight_reset)

Finally, here is the model specification itself:

In [ ]:
class MPNN(GraphSequenceModel):
    """An MPNN with a `num_layers` message passing layers, then an MLP."""

    def _layer_sequence(self, num_layers=16):

        # The sequence of layers
        sequence = []

        # Add `num_layers` message passing layers
        for i in range(num_layers):
            sequence.append((GCNConv(50, 50), f"x, edge_index -> x"))
            sequence.append(nn.ReLU())
        
        # A global mean pool layer
        sequence.append((GlobalMeanPool(), f"x, batch -> x"))

        # Add an MLP at the end
        sequence.extend([
            (nn.Linear(50, 70), "x -> x"),
            nn.ReLU(),
            (nn.Linear(70, 25), "x -> x"),
            nn.ReLU(),
            (nn.Linear(25,2), "x -> x")
        ])
        
        return sequence

Let's now instantiate the model:

In [ ]:
model1 = MPNN(num_layers=16).to(device)

## **Part 1.3: Training, testing and cross-validation functions**
This is the generic training loop, which does one epoch-worth of training:


In [ ]:
def train_epoch(dataloader, model, loss_fn, optimiser):
    """Do one epoch-worth of training."""
    
    # Put the model in training mode
    model.train()
    
    # The number of datapoints
    size = len(dataloader.dataset)
    
    # Loop over each batch of datapoints
    for data in dataloader:
        data.to(device)

        # Set all the gradients to zero
        optimiser.zero_grad()
        
        # Make a prediction using the current parameters
        pred = model(data)
        
        # Compute the loss for this prediction
        loss = loss_fn(pred, data.y)
        
        # Propagate the loss backwards to compute the gradients
        loss.backward()
        
        # Do one step of optimisation
        optimiser.step()

This function does a full train on the data:

In [ ]:
def train(train_dataloader, test_dataloader, model, loss_fn, optimiser, 
          epochs=200, output_every=20):
    """Train a model for a certain number of epochs."""

    # Loop through the epochs
    for t in range(1, epochs+1):

        # Do the training for this epoch
        train_epoch(train_dataloader, model, loss_fn, optimiser)

        # Output the accuracy of the model every so often
        if output_every is not None and t % output_every == 0:
            print(f"Epoch {t}")
            print("----------------------------")
            print(f"Train accuracy: {test(train_dataloader, model):%}")
            print(f"Test accuracy: {test(test_dataloader, model):%}")
            print()

This function tests the model on the data, and returns the accuracy:

In [ ]:
def test(dataloader, model):
    """Test a model on some data."""
    
    # Put the model in evaluation mode
    model.eval()
    
    # Get the number of datapoints
    size = len(dataloader.dataset)

    # The number of correct predictions
    correct = 0

    # We don't want to be computing the gradients
    with torch.no_grad():

        # Loop through the minibatches
        for data in dataloader:
            data.to(device)
            # Compute the model predictions
            pred = model(data)

            # Update with the number of correct predictions
            correct += (pred.argmax(1) == data.y).count_nonzero()

    # Compute the accuracy for the whole dataset and return it
    return correct / len(dataloader.dataset)

This function performs cross-validation on the dataset:

In [ ]:
def cross_validate(dataset, model, loss_fn, optimiser, num_splits=5,
                   batch_size=32, epochs=200, output_every=20):
    """Use k-fold cross validation to evaluate a model on a dataset.
    
    Assumes that the dataset is ordered into consecutive pairs, and then 
    shuffles and splits the data so that both elements of each pair get into
    the same split.
    """

    # Get the number of graphs and number of pairs
    size = len(dataset)
    num_pairs = size / 2

    # Construct a permuter which keeps paired graphs together
    pair_permuter = rng.permutation(np.arange(num_pairs)) * 2
    graph_permuter = np.empty((size,), dtype=int)
    graph_permuter[0::2] = pair_permuter
    graph_permuter[1::2] = pair_permuter + 1

    # Use the permuter to shuffle the dataset
    shuffled_dataset = []
    for i in graph_permuter.flat:
        shuffled_dataset.append(dataset[i])

    # Arrays to store the train and test accuracies for each fold
    train_accuracies = np.zeros(num_splits)
    test_accuracies = np.zeros(num_splits)
    
    # Loop over the folds
    for fold in range(num_splits): 

        # Print a header to mark this fold
        print(f"Fold {fold+1}")
        print("============================")
        print()

        # Calculate the current fold segment indices
        index_min = int(((num_pairs * fold) // num_splits) * 2)
        index_max = int(((num_pairs * (fold+1)) // num_splits) * 2)

        # print(index_min, index_max)

        # Split into train and test datasets
        train_dataset = (shuffled_dataset[:index_min] 
                         + shuffled_dataset[index_max:])
        test_dataset = shuffled_dataset[index_min:index_max]

        # Turn these into torch_geometric dataloaders
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
        test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

        # Reset the parameter of the model before training
        model.reset_parameters()

        # Train with these
        train(train_dataloader, test_dataloader, model, loss_fn, optimiser, 
              epochs, output_every)

        # Record the test and train accuracies for the trained model
        train_accuracies[fold] = test(train_dataloader, model)
        test_accuracies[fold] = test(test_dataloader, model)
    
    # Print the Train and test accuracies for each fold
    print(f"{num_splits}-fold validation summary")
    print("============================")
    for fold in range(num_splits):
        print(f"Fold {fold+1}. Train: {train_accuracies[fold]:09.5%} "
              f"Test: {test_accuracies[fold]:09.5%}")


## Task 1.4: Training the model

Specify loss and optimiser and train the model using the `cross_validate` function. Use a learning rate of 1e-3 and 100 epochs. What results do you get? Is it possible to improve the architecture to get a better test accuracy? Why?

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model1.parameters(), 
                             lr=1e-3)
cross_validate(dataset, model1, loss_fn, optimiser, epochs=100)

Fold 1

Epoch 20
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 40
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 60
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 80
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 100
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Fold 2

Epoch 20
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 40
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 60
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 80
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 100
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Fold 3

Epoch 20
----------------------------
Train accuracy: 

KeyboardInterrupt: ignored

# **Part 2: Introducing randomization**


## **Task 2.1: Adding a RandomNodeInitialiser**
Define a layer to be placed before of the GCNConv layers that introduces randomness on half of the features of each nodes. You can use `torch.randn` to generate noise. Look at the function `.masked_scatter`.


In [ ]:
class RandomNodeInitialiser(nn.Module):
    """Randomises the first `num_rand_features` input features."""

    def __init__(self, num_features, num_rand_features):
        super().__init__()
        self.mask = torch.hstack([torch.ones(num_rand_features,dtype=torch.bool),
                                  torch.zeros(num_features - num_rand_features,dtype=torch.bool)])
        print(self.mask)
        self.mask = self.mask.to(device)

    def forward(self, x):
        noise = torch.randn(x.size()).to(device)
        x = x.masked_scatter(self.mask, noise)
        return x

    def extra_repr(self):
        return f"{self.num_rand_features}"

Here is the model modified to modify 25 input features:

In [ ]:

class MPNNRand(MPNN):

    def _layer_sequence(self, num_layers=16, rni_num_features=25):
         sequence = super()._layer_sequence(num_layers)
         random_layer = RandomNodeInitialiser(50, rni_num_features)
         sequence = [(random_layer, 'x -> x')] + sequence

         return sequence


Let's instantiate the model:

In [ ]:
model2 = MPNNRand().to(device)

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])


## **Task 2.2: Training the model**
Specify loss and optimiser and train the model using the cross_validate function. Use a learning rate of 1e-4 and train for 2000 epochs. What results do you get?

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model2.parameters(), 
                             lr=1e-4)
cross_validate(dataset, model2, loss_fn, optimiser, epochs=2000)

Fold 1

Epoch 20
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 40
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 60
----------------------------
Train accuracy: 53.409094%
Test accuracy: 68.181819%

Epoch 80
----------------------------
Train accuracy: 50.000000%
Test accuracy: 50.000000%

Epoch 100
----------------------------
Train accuracy: 52.272731%
Test accuracy: 50.000000%

Epoch 120
----------------------------
Train accuracy: 50.000000%
Test accuracy: 45.454547%

Epoch 140
----------------------------
Train accuracy: 46.590909%
Test accuracy: 45.454547%

Epoch 160
----------------------------
Train accuracy: 57.954550%
Test accuracy: 63.636363%

Epoch 180
----------------------------
Train accuracy: 50.000000%
Test accuracy: 45.454547%

Epoch 200
----------------------------
Train accuracy: 63.636363%
Test accuracy: 68.181819%

Epoch 220
----------------------------
Train accuracy: 61.363637%


## **Part 2.3 Ablation studies (Optional)**

### **Task 2.3.1**
Train a model with only 4 and 8 message passage layers. What do you observe? Why?



In [ ]:
pass

In [ ]:
pass

Both of these perform worse than the 16-layer model, with the 8-layer slightly better than the 4-layer. The reason is that there are cycles up to 15 nodes long. With fewer message passing layers than nodes, a message can't travel all the way round a cycle and return to the origin node. So a 4-layer message passing network can't tell the difference between an 8-cycle and a 9-cycle. In fact, all long cycles appear to be infinitely long.

### **Task 2.3.2**
Train a model with only 1 modified feature per node (i.e., the feauters affected by the `RandomNodeInitialiser`).

In [ ]:
pass